## Import Library

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import time

import os, json, pickle, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.utils import compute_class_weight
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import (
    Embedding, LSTM, GRU, Dense, Dropout,
    Bidirectional, GlobalMaxPool1D, Conv1D,
    MaxPooling1D, BatchNormalization, Flatten, GlobalAveragePooling1D,
)
from tensorflow.keras.callbacks import TensorBoard
import datetime
from collections import Counter

from tensorflow.keras.layers import Layer
from transformers import BertTokenizer, TFBertModel


In [ ]:
# Inisiasi variabel Global
RANDOM_SEED   = 42
NUM_WORDS     = 10000
MAX_LENGTH    = 200
EMBED_DIM     = 128
EPOCH         = 50
BATCH_SIZE    = 32
OUTPUT_DIR    = './models'
LOGS_DIR      = './logs'
DATASET_PATH  = 'Emotion Dataset Merged.xlsx'  
SAVED_MODEL_DIR = './models/saved_model'         

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)
os.makedirs(SAVED_MODEL_DIR, exist_ok=True)
LOG_DIR = os.path.join(LOGS_DIR, datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))

## Load Dataset

In [ ]:
df =pd.read_excel(DATASET_PATH)
print(f'Shape : {df.shape}')
df.head()

## EDA

In [ ]:
df.info()
print('\nMissing values:')
print(df.isnull().sum())
print(f'\nDuplikat : {df.duplicated().sum()}')
df.describe()

In [ ]:

#Distribusi panjang teks
df['input_length'] = df['input_clean'].str.split().str.len()
df['label_length'] = df['predicted_emotion'].str.split().str.len()
print(df[['input_length', 'label_length']].describe())

#Cek duplikat input (input sama, label berbeda)
print(f"Input unik: {df['input_clean'].nunique()} dari {len(df)} total")

#Cek label noise (confidence rendah)
print(df[df['emotion_confidence'] < 0.5]['predicted_emotion'].value_counts())


In [ ]:
# ── Distribusi Kelas ─────────────────────────────────────────
label_counts = df['predicted_emotion'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(label_counts.index, label_counts.values,
            color=['#e74c3c', '#3498db', '#2ecc71'])
axes[0].set_title('Distribusi Kelas Emosi')
axes[0].set_xlabel('Emosi')
axes[0].set_ylabel('Jumlah')
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(label_counts.values, labels=label_counts.index,
            autopct='%1.1f%%', colors=['#e74c3c', '#3498db', '#2ecc71'],
            startangle=90)
axes[1].set_title('Proporsi Kelas')

plt.tight_layout()
plt.show()
print(label_counts)

## Split Data

In [ ]:
def compute_healing_score(row):
    return row['acceptance_prob'] - (row['anger_prob'] + row['anxiety_prob']) / 2

df['healing_score'] = df.apply(compute_healing_score, axis=1)

print("healing_score stats:")
print(df['healing_score'].describe())

In [ ]:
X = df['input_clean'].fillna('').astype(str)
y = df['predicted_emotion']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

NUM_CLASSES = len(le.classes_)

print('\nLabel encoding:')
for i, cls in enumerate(le.classes_):
    print(f'  {cls:12s} → {i}')

y_healing = df['healing_score'].values.astype(np.float32)
print(f'\nHealing score — min: {y_healing.min():.4f}, max: {y_healing.max():.4f}, mean: {y_healing.mean():.4f}')

with open(os.path.join(OUTPUT_DIR, 'label_encoder.pkl'), 'wb') as f:
    pickle.dump(le, f)
print("✓ Label encoder disimpan.")

In [ ]:
X_train, X_temp, y_train, y_temp, yh_train, yh_temp = train_test_split(
    X, y_encoded, y_healing,
    test_size=0.2, random_state=RANDOM_SEED, stratify=y_encoded
)

X_val, X_test, y_val, y_test, yh_val, yh_test = train_test_split(
    X_temp, y_temp, yh_temp,
    test_size=0.5, random_state=RANDOM_SEED, stratify=y_temp
)

print(f'\nData split:')
print(f'  Train set : {len(X_train):,} samples ({len(X_train)/len(X)*100:.1f}%)')
print(f'  Val set   : {len(X_val):,} samples ({len(X_val)/len(X)*100:.1f}%)')
print(f'  Test set  : {len(X_test):,} samples ({len(X_test)/len(X)*100:.1f}%)')

print("Distribusi train:", Counter(y_train))
print("Distribusi val:  ", Counter(y_val))
print("Distribusi test: ", Counter(y_test))


## Tokenisasi dan Padding

In [ ]:
tokenizer = Tokenizer(num_words=NUM_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

vocab_size = min(len(tokenizer.word_index) + 1, NUM_WORDS)
print(f'Vocab size (aktual) : {len(tokenizer.word_index):,}')
print(f'Vocab size (capped) : {vocab_size:,}')

def texts_to_padded(texts):
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=MAX_LENGTH, padding='post', truncating='post')

X_train_seq = texts_to_padded(X_train)
X_val_seq   = texts_to_padded(X_val)
X_test_seq  = texts_to_padded(X_test)

print(f'\nShape setelah padding:')
print(f'  Train : {X_train_seq.shape}')
print(f'  Val   : {X_val_seq.shape}')
print(f'  Test  : {X_test_seq.shape}')

with open(os.path.join(OUTPUT_DIR, 'tokenizer.pkl'), 'wb') as f:
    pickle.dump(tokenizer, f)
print(" Tokenizer disimpan.")

### Pembuatan Tf.data.dataset

In [ ]:
train_ds = (
    tf.data.Dataset
    .from_tensor_slices((X_train_seq, (y_train, yh_train)))
    .shuffle(len(X_train_seq), seed=RANDOM_SEED)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset
    .from_tensor_slices((X_val_seq, (y_val, yh_val)))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_ds = (
    tf.data.Dataset
    .from_tensor_slices((X_test_seq, (y_test, yh_test)))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight = {i: w for i, w in enumerate(class_weights_arr)}

print(f"  train_ds : {len(list(train_ds))} batch")
print(f"  val_ds   : {len(list(val_ds))} batch")
print(f"  test_ds  : {len(list(test_ds))} batch")
print(f"\nClass weight: {class_weight}")

## Ekstraksi Fitur

### TF-IDF

In [ ]:
tfidf_vectorizer = TfidfVectorizer(max_features=5000, min_df=5, max_df=0.8, ngram_range=(1, 2))
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_val_tfidf   = tfidf_vectorizer.transform(X_val)
X_test_tfidf  = tfidf_vectorizer.transform(X_test)
print(f'TF-IDF matrix shape: {X_train_tfidf.shape}')

smote_tfidf = SMOTE(random_state=RANDOM_SEED)
X_train_tfidf_smote, y_train_tfidf_smote = smote_tfidf.fit_resample(X_train_tfidf, y_train)

print('\nSMOTE TF-IDF — distribusi setelah resampling:')
unique, counts = np.unique(y_train_tfidf_smote, return_counts=True)
for cls, cnt in zip(le.classes_, counts):
    print(f'  {cls:12s}: {cnt}')
print(f'  Total: {len(y_train_tfidf_smote):,}')

### CountVectorizer

In [ ]:
count_vectorizer = CountVectorizer(max_features=5000, min_df=5, max_df=0.8, ngram_range=(1, 2))
X_train_count = count_vectorizer.fit_transform(X_train)
X_val_count   = count_vectorizer.transform(X_val)
X_test_count  = count_vectorizer.transform(X_test)
print(f'Count matrix shape: {X_train_count.shape}')

smote_count = SMOTE(random_state=RANDOM_SEED)
X_train_count_smote, y_train_count_smote = smote_count.fit_resample(X_train_count, y_train)

print('\nSMOTE Count — distribusi setelah resampling:')
unique, counts = np.unique(y_train_count_smote, return_counts=True)
for cls, cnt in zip(le.classes_, counts):
    print(f'  {cls:12s}: {cnt}')
print(f'  Total: {len(y_train_count_smote):,}')

## Modeling

### Custom Layer

In [ ]:
class CustomEmbeddingLayer(tf.keras.layers.Layer):
    def __init__(self, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.embed_dim  = embed_dim

    def build(self, input_shape):
        self.embedding_matrix = self.add_weight(
            name='embedding_matrix',
            shape=(self.vocab_size, self.embed_dim),
            initializer=tf.keras.initializers.TruncatedNormal(stddev=0.02),
            trainable=True
        )
        super().build(input_shape)

    def call(self, token_ids):
        x    = tf.nn.embedding_lookup(self.embedding_matrix, token_ids)
        mask = tf.cast(tf.not_equal(token_ids, 0), tf.float32)
        mask = tf.expand_dims(mask, -1)
        return x * mask

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'vocab_size': self.vocab_size, 'embed_dim': self.embed_dim})
        return cfg


class CustomDenseLayer(tf.keras.layers.Layer):
    def __init__(self, units=32, activation='relu', dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.units        = units
        self.activation   = tf.keras.activations.get(activation)
        self.dropout_rate = dropout_rate

    def build(self, input_shape):
        self.W = self.add_weight(
            name='kernel',
            shape=(input_shape[-1], self.units),
            initializer='glorot_uniform',
            trainable=True
        )
        self.b = self.add_weight(
            name='bias',
            shape=(self.units,),
            initializer='zeros',
            trainable=True
        )
        self.dropout = tf.keras.layers.Dropout(self.dropout_rate)
        super().build(input_shape)

    def call(self, inputs, training=False):
        x = tf.matmul(inputs, self.W) + self.b
        x = self.activation(x)
        x = self.dropout(x, training=training)
        return x

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'units': self.units, 'dropout_rate': self.dropout_rate})
        return cfg


class AttentionLayer(tf.keras.layers.Layer):
    def __init__(self, units=64, **kwargs):
        super().__init__(**kwargs)
        self.units = units

    def build(self, input_shape):
        self.W_query = self.add_weight(
            name='W_query',
            shape=(input_shape[-1], self.units),
            initializer='glorot_uniform', trainable=True
        )
        self.W_key = self.add_weight(
            name='W_key',
            shape=(input_shape[-1], self.units),
            initializer='glorot_uniform', trainable=True
        )
        self.V = self.add_weight(
            name='V',
            shape=(self.units, 1),
            initializer='glorot_uniform', trainable=True
        )
        self.b = self.add_weight(
            name='bias',
            shape=(self.units,),
            initializer='zeros', trainable=True
        )
        super().build(input_shape)

    def call(self, inputs):
        query   = tf.tanh(tf.matmul(inputs, self.W_query) + self.b)
        key     = tf.tanh(tf.matmul(inputs, self.W_key))
        score   = tf.matmul(query + key, self.V)
        weights = tf.nn.softmax(score, axis=1)
        context = tf.reduce_sum(inputs * weights, axis=1)
        return context

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'units': self.units})
        return cfg


class CustomConv1DLayer(tf.keras.layers.Layer):
    def __init__(self, filters=128, kernel_size=5,
                 activation='relu', dropout_rate=0.3, **kwargs):
        super().__init__(**kwargs)
        self.filters      = filters
        self.kernel_size  = kernel_size
        self.activation   = tf.keras.activations.get(activation)
        self.dropout_rate = dropout_rate

    def build(self, input_shape):
        self.conv = tf.keras.layers.Conv1D(
            filters=self.filters,
            kernel_size=self.kernel_size,
            padding='same',
            use_bias=False
        )
        self.bn      = tf.keras.layers.BatchNormalization()
        self.dropout = tf.keras.layers.Dropout(self.dropout_rate)
        super().build(input_shape)

    def call(self, inputs, training=False):
        x = self.conv(inputs)
        x = self.bn(x, training=training)
        x = self.activation(x)
        x = self.dropout(x, training=training)
        return x

    def get_config(self):
        cfg = super().get_config()
        cfg.update({
            'filters':      self.filters,
            'kernel_size':  self.kernel_size,
            'dropout_rate': self.dropout_rate
        })
        return cfg

### Custom LossFN

In [ ]:
class CustomLossFunction(tf.keras.losses.Loss):

    def __init__(self, mse_weight=0.01, **kwargs):
        super().__init__(**kwargs)
        self.mse_weight = mse_weight

    def call(self, y_true, y_pred):
        ce_loss = tf.keras.losses.sparse_categorical_crossentropy(
            y_true, y_pred, from_logits=False
        )
        ce_loss = tf.reduce_mean(ce_loss)

        y_true_onehot = tf.one_hot(
            tf.cast(y_true, tf.int32),
            depth=tf.shape(y_pred)[-1]
        )
        mse_loss = tf.reduce_mean(tf.square(y_true_onehot - y_pred))

        return ce_loss + self.mse_weight * mse_loss

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'mse_weight': self.mse_weight})
        return cfg


class MultiOutputLoss(tf.keras.losses.Loss):

    def __init__(self, emotion_weight=0.7, healing_weight=0.3,
                 mse_weight=0.01, huber_delta=0.5, **kwargs):
        super().__init__(**kwargs)
        self.emotion_weight = emotion_weight
        self.healing_weight = healing_weight
        self.mse_weight     = mse_weight
        self.huber_delta    = huber_delta
        self._emotion_loss  = CustomLossFunction(mse_weight=mse_weight)
        self._huber         = tf.keras.losses.Huber(delta=huber_delta)

    def call(self, y_true, y_pred):
        y_emotion,    y_healing    = y_true
        pred_emotion, pred_healing = y_pred

        loss_emotion = self._emotion_loss(y_emotion, pred_emotion)
        loss_healing = self._huber(
            tf.expand_dims(y_healing, -1),
            pred_healing
        )
        return self.emotion_weight * loss_emotion + self.healing_weight * loss_healing

    def get_config(self):
        cfg = super().get_config()
        cfg.update({
            'emotion_weight': self.emotion_weight,
            'healing_weight': self.healing_weight,
            'mse_weight':     self.mse_weight,
            'huber_delta':    self.huber_delta,
        })
        return cfg


class FocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0, alpha=0.25, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        ce    = tf.keras.losses.sparse_categorical_crossentropy(
            y_true, y_pred, from_logits=False
        )
        pt    = tf.exp(-ce)
        focal = self.alpha * tf.pow(1.0 - pt, self.gamma) * ce
        return tf.reduce_mean(focal)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'gamma': self.gamma, 'alpha': self.alpha})
        return cfg


class LabelSmoothingLoss(tf.keras.losses.Loss):
    def __init__(self, num_classes, smoothing=0.1, **kwargs):
        super().__init__(**kwargs)
        self.num_classes = num_classes
        self.smoothing   = smoothing

    def call(self, y_true, y_pred):
        y_true_onehot = tf.one_hot(
            tf.cast(y_true, tf.int32),
            depth=self.num_classes
        )
        smooth_labels = (
            y_true_onehot * (1.0 - self.smoothing) +
            (self.smoothing / self.num_classes)
        )
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0)
        loss   = -tf.reduce_sum(smooth_labels * tf.math.log(y_pred), axis=-1)
        return tf.reduce_mean(loss)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'num_classes': self.num_classes, 'smoothing': self.smoothing})
        return cfg


### Custom CallBack


In [ ]:
class CustomEarlyStopping(tf.keras.callbacks.Callback):

    def __init__(self, patience=5, min_delta=0.001):
        super().__init__()
        self.patience  = patience
        self.min_delta = min_delta
        self.wait      = 0
        self.best_loss = np.inf

    def on_train_begin(self, logs=None):
        self.wait      = 0
        self.best_loss = np.inf

    def on_epoch_end(self, epoch, logs=None):
        current = logs.get('val_loss')
        if current is None:
            return
        if current < self.best_loss - self.min_delta:
            self.best_loss = current
            self.wait      = 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                print(f"\n[EarlyStopping] Berhenti di epoch {epoch+1} "
                      f"— val_loss tidak membaik selama {self.patience} epoch.")
                self.model.stop_training = True


class StopAtAccuracy(tf.keras.callbacks.Callback):

    def __init__(self, target_acc=0.95):
        super().__init__()
        self.target_acc = target_acc

    def on_epoch_end(self, epoch, logs=None):
        acc = logs.get('accuracy')
        if acc is not None and acc >= self.target_acc:
            print(f"\n[StopAtAccuracy] Target {self.target_acc:.0%} "
                  f"tercapai di epoch {epoch+1} (acc={acc:.4f})")
            self.model.stop_training = True


class CustomModelCheckpoint(tf.keras.callbacks.Callback):

    def __init__(self, filepath, monitor='val_loss', save_best_only=True):
        super().__init__()
        self.filepath       = filepath
        self.monitor        = monitor
        self.save_best_only = save_best_only
        self.best = np.inf if 'loss' in monitor else -np.inf

    def on_epoch_end(self, epoch, logs=None):
        current = logs.get(self.monitor)
        if current is None:
            return
        is_better = (
            current < self.best if 'loss' in self.monitor
            else current > self.best
        )
        if not self.save_best_only or is_better:
            print(f"\n[Checkpoint] Simpan model → {self.filepath} "
                  f"({self.monitor}={current:.4f})")
            self.model.save(self.filepath)
            self.best = current


class CustomTensorBoard(tf.keras.callbacks.TensorBoard):

    def __init__(self, log_dir=None, **kwargs):
        log_dir = log_dir or LOG_DIR
        super().__init__(log_dir=log_dir, **kwargs)

    def on_epoch_end(self, epoch, logs=None):
        super().on_epoch_end(epoch, logs)
        if logs:
            print(f"  [TB] Epoch {epoch+1:03d} — "
                  f"loss: {logs.get('loss', 0):.4f}  "
                  f"acc: {logs.get('accuracy', 0):.4f}  "
                  f"val_loss: {logs.get('val_loss', 0):.4f}  "
                  f"val_acc: {logs.get('val_accuracy', 0):.4f}")


class CustomLearningRateScheduler(tf.keras.callbacks.Callback):

    def __init__(self, initial_lr=0.001, decay_factor=0.5, step_size=5):
        super().__init__()
        self.initial_lr   = initial_lr
        self.decay_factor = decay_factor
        self.step_size    = step_size

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.step_size == 0:
            new_lr = self.initial_lr * (
                self.decay_factor ** ((epoch + 1) // self.step_size)
            )
            self.model.optimizer.learning_rate.assign(new_lr)
            print(f"\n[LRScheduler] Epoch {epoch+1} — LR → {new_lr:.6f}")


class TrainingSummaryLogger(tf.keras.callbacks.Callback):

    def __init__(self, save_path=None):
        super().__init__()
        self.save_path  = save_path
        self.start_time = None

    def on_train_begin(self, logs=None):
        self.start_time = time.time()
        total_batches   = self.params.get('steps', '?')
        print(f"\n[SummaryLogger] Training dimulai — "
              f"{total_batches} batch/epoch\n")

    def on_train_end(self, logs=None):
        elapsed  = time.time() - self.start_time
        avg_loss = logs.get('loss', 'N/A')
        print(f"\n[SummaryLogger] Selesai dalam {elapsed:.1f} detik")
        print(f"[SummaryLogger] Loss akhir : {avg_loss}")
        if self.save_path:
            self.model.save(self.save_path)
            print(f"[SummaryLogger] Model disimpan → {self.save_path}")


class OverfittingDetector(tf.keras.callbacks.Callback):

    def __init__(self, threshold=0.15):
        super().__init__()
        self.threshold = threshold

    def on_epoch_end(self, epoch, logs=None):
        train_loss = logs.get('loss', 0)
        val_loss   = logs.get('val_loss', 0)
        gap        = val_loss - train_loss
        if gap > self.threshold:
            print(f"\n[OverfittingDetector] Epoch {epoch+1} — "
                  f"gap={gap:.4f} "
                  f"(train={train_loss:.4f}, val={val_loss:.4f})")



### Custom TrainLOOp

In [ ]:
class customTrainingLoop:
    def __init__(
        self,
        model,
        model_name,
        optimizer,
        loss_fn,
        log_dir      = None,
        class_weight = None,
        callbacks    = None,
    ):
        self.model        = model
        self.model_name   = model_name
        self.optimizer    = optimizer
        self.loss_fn      = loss_fn
        self.log_dir      = log_dir or LOG_DIR
        self.class_weight = class_weight
        self.callbacks    = callbacks or []

        self.history = {
            'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': [],
            'mae': [], 'val_mae': [],
        }

        self.best_val_loss   = np.inf
        self.checkpoint_path = os.path.join(OUTPUT_DIR, f'{self.model_name}_best.keras')

        self.train_writer = tf.summary.create_file_writer(
            os.path.join(self.log_dir, self.model_name, 'train')
        )
        self.val_writer = tf.summary.create_file_writer(
            os.path.join(self.log_dir, self.model_name, 'val')
        )

        self.train_loss_metric = tf.keras.metrics.Mean(name='train_loss')
        self.train_acc_metric  = tf.keras.metrics.SparseCategoricalAccuracy(name='train_acc')
        self.train_mae_metric  = tf.keras.metrics.MeanAbsoluteError(name='train_mae')
        self.val_loss_metric   = tf.keras.metrics.Mean(name='val_loss')
        self.val_acc_metric    = tf.keras.metrics.SparseCategoricalAccuracy(name='val_acc')
        self.val_mae_metric    = tf.keras.metrics.MeanAbsoluteError(name='val_mae')


   
    @tf.function
    def _train_step(self, X_batch, y_emotion, y_healing):
        with tf.GradientTape() as tape:
            pred_emotion, pred_healing = self.model(X_batch, training=True)
            loss = self.loss_fn(
                (y_emotion, y_healing),
                (pred_emotion, pred_healing)
            )
            if self.class_weight is not None:
                weights = tf.gather(
                    tf.constant(list(self.class_weight.values()), dtype=tf.float32),
                    tf.cast(y_emotion, tf.int32)
                )
                loss = loss * tf.reduce_mean(weights)

        grads = tape.gradient(loss, self.model.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.model.trainable_variables))

        self.train_loss_metric.update_state(loss)
        self.train_acc_metric.update_state(y_emotion, pred_emotion)
        self.train_mae_metric.update_state(tf.expand_dims(y_healing, -1), pred_healing)


    @tf.function
    def _val_step(self, X_batch, y_emotion, y_healing):
        pred_emotion, pred_healing = self.model(X_batch, training=False)
        loss = tf.reduce_mean(
            self.loss_fn(
                (y_emotion, y_healing),
                (pred_emotion, pred_healing)
            )
        )
        self.val_loss_metric.update_state(loss)
        self.val_acc_metric.update_state(y_emotion, pred_emotion)
        self.val_mae_metric.update_state(tf.expand_dims(y_healing, -1), pred_healing)


    def fit(self, train_ds, val_ds, epochs):
        print(f"\n{'═'*60}")
        print(f"  Model     : {self.model_name}")
        print(f"  Epochs    : {epochs}")
        print(f"  Optimizer : {self.optimizer.__class__.__name__}")
        print(f"  Loss      : {self.loss_fn.__class__.__name__}")
        print(f"  LR        : {float(self.optimizer.learning_rate):.6f}")
        print(f"  Callbacks : {[c.__class__.__name__ for c in self.callbacks]}")
        print(f"{'═'*60}\n")

        for cb in self.callbacks:
            cb.model = self.model
        for cb in self.callbacks:
            if hasattr(cb, 'on_train_begin'):
                cb.on_train_begin(logs={})

        self.total_start = time.time()
        t_loss = 0

        for epoch in range(epochs):
            epoch_start = time.time()

            for X_batch, (y_emotion, y_healing) in train_ds:
                self._train_step(X_batch, y_emotion, y_healing)

            for X_batch, (y_emotion, y_healing) in val_ds:
                self._val_step(X_batch, y_emotion, y_healing)

            t_loss  = self.train_loss_metric.result().numpy()
            t_acc   = self.train_acc_metric.result().numpy()
            t_mae   = self.train_mae_metric.result().numpy()
            v_loss  = self.val_loss_metric.result().numpy()
            v_acc   = self.val_acc_metric.result().numpy()
            v_mae   = self.val_mae_metric.result().numpy()
            elapsed = time.time() - epoch_start

            self.history['loss'].append(t_loss)
            self.history['accuracy'].append(t_acc)
            self.history['mae'].append(t_mae)
            self.history['val_loss'].append(v_loss)
            self.history['val_accuracy'].append(v_acc)
            self.history['val_mae'].append(v_mae)

            with self.train_writer.as_default():
                tf.summary.scalar('loss',     t_loss, step=epoch)
                tf.summary.scalar('accuracy', t_acc,  step=epoch)
                tf.summary.scalar('mae',      t_mae,  step=epoch)

            with self.val_writer.as_default():
                tf.summary.scalar('loss',     v_loss, step=epoch)
                tf.summary.scalar('accuracy', v_acc,  step=epoch)
                tf.summary.scalar('mae',      v_mae,  step=epoch)

            print(f"Epoch {epoch+1:03d}/{epochs} ({elapsed:.1f}s) — "
                  f"loss: {t_loss:.4f}  acc: {t_acc:.4f}  mae: {t_mae:.4f}  "
                  f"val_loss: {v_loss:.4f}  val_acc: {v_acc:.4f}  val_mae: {v_mae:.4f}")

            gap = v_loss - t_loss
            if gap > 0.15:
                print(f"Overfitting — gap: {gap:.4f}")

            if v_loss < self.best_val_loss:
                self.best_val_loss = v_loss
                self.model.save(self.checkpoint_path)
                print(f"✓ Best model saved (val_loss={v_loss:.4f})")

            self.train_loss_metric.reset_states()
            self.train_acc_metric.reset_states()
            self.train_mae_metric.reset_states()
            self.val_loss_metric.reset_states()
            self.val_acc_metric.reset_states()
            self.val_mae_metric.reset_states()

            logs = {
                'loss': t_loss, 'accuracy': t_acc, 'mae': t_mae,
                'val_loss': v_loss, 'val_accuracy': v_acc, 'val_mae': v_mae,
            }
            for cb in self.callbacks:
                if hasattr(cb, 'on_epoch_end'):
                    cb.on_epoch_end(epoch, logs=logs)

            if getattr(self.model, 'stop_training', False):
                print(f"⛔ Training dihentikan oleh callback di epoch {epoch+1}")
                break

        total_elapsed = time.time() - self.total_start

        for cb in self.callbacks:
            if hasattr(cb, 'on_train_end'):
                cb.on_train_end(logs={'loss': t_loss})

        print(f"\n{'═'*60}")
        print(f"  Selesai       : {self.model_name}")
        print(f"  Total waktu   : {total_elapsed:.1f} detik")
        print(f"  Best val_loss : {self.best_val_loss:.4f}")
        print(f"  Checkpoint    : {self.checkpoint_path}")
        print(f"{'═'*60}\n")


    def evaluate(self, test_ds, y_test, yh_test, le):
        print(f"\n{'─'*60}")
        print(f"  Evaluasi : {self.model_name}")
        print(f"{'─'*60}")

        y_pred_emotion_list = []
        y_pred_healing_list = []

        for X_batch, _ in test_ds:
            pred_e, pred_h = self.model(X_batch, training=False)
            y_pred_emotion_list.append(pred_e.numpy())
            y_pred_healing_list.append(pred_h.numpy())

        y_pred_emotion_probs = np.concatenate(y_pred_emotion_list, axis=0)
        y_pred_healing       = np.concatenate(y_pred_healing_list, axis=0).flatten()
        y_pred_emotion       = np.argmax(y_pred_emotion_probs, axis=1)

        acc      = accuracy_score(y_test, y_pred_emotion)
        f1_macro = f1_score(y_test, y_pred_emotion, average='macro')
        f1_w     = f1_score(y_test, y_pred_emotion, average='weighted')
        mae      = np.mean(np.abs(yh_test - y_pred_healing))

        print(f"  Accuracy   : {acc:.4f}  {'✓' if acc >= 0.85 else '✗'} target > 0.85")
        print(f"  F1 Macro   : {f1_macro:.4f}")
        print(f"  F1 Weighted: {f1_w:.4f}")
        print(f"  MAE Healing: {mae:.4f}  {'✓ < 0.2' if mae < 0.2 else '✗ > 0.2'}")
        print(f"\n{classification_report(y_test, y_pred_emotion, target_names=le.classes_)}")

        cm = confusion_matrix(y_test, y_pred_emotion)
        plt.figure(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=le.classes_, yticklabels=le.classes_)
        plt.title(f'Confusion Matrix — {self.model_name}')
        plt.ylabel('True'); plt.xlabel('Predicted')
        plt.tight_layout(); plt.show()

        return {
            'accuracy':    acc,
            'f1_macro':    f1_macro,
            'f1_weighted': f1_w,
            'mae_healing': mae,
        }


    def plot_history(self):
        epochs = range(1, len(self.history['loss']) + 1)

        fig, axes = plt.subplots(1, 3, figsize=(18, 4))
        fig.suptitle(f'Training History — {self.model_name}', fontsize=13)

        axes[0].plot(epochs, self.history['loss'],     label='Train Loss')
        axes[0].plot(epochs, self.history['val_loss'], label='Val Loss')
        axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

        axes[1].plot(epochs, self.history['accuracy'],     label='Train Acc')
        axes[1].plot(epochs, self.history['val_accuracy'], label='Val Acc')
        axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

        axes[2].plot(epochs, self.history['mae'],     label='Train MAE')
        axes[2].plot(epochs, self.history['val_mae'], label='Val MAE')
        axes[2].set_title('MAE Healing Score'); axes[2].legend(); axes[2].grid(True, alpha=0.3)

        plt.tight_layout(); plt.show()


    def predict(self, text, tokenizer, le, max_length=MAX_LENGTH):
        seq    = tokenizer.texts_to_sequences([text])
        padded = pad_sequences(seq, maxlen=max_length, padding='post', truncating='post')
        pred_e, pred_h = self.model.predict(padded, verbose=0)
        probs  = pred_e[0]
        idx    = np.argmax(probs)

        print(f"  Input         : {text}")
        print(f"  Emosi         : {le.classes_[idx]} ({probs[idx]:.2%})")
        print(f"  Healing Score : {float(pred_h[0][0]):.4f}")
        print(f"  Semua emosi   : { {le.classes_[i]: f'{p:.2%}' for i, p in enumerate(probs)} }")

        return le.classes_[idx], probs[idx], float(pred_h[0][0])



### Model Function

In [ ]:
def model_LSTM():
    inputs = tf.keras.Input(shape=(MAX_LENGTH,), name='input')
    x      = CustomEmbeddingLayer(vocab_size, EMBED_DIM, name='custom_embedding')(inputs)
    x      = LSTM(128, return_sequences=True, name='lstm_1')(x)
    x      = Dropout(0.4, name='dropout_1')(x)
    x      = LSTM(64, name='lstm_2')(x)
    x      = Dropout(0.4, name='dropout_2')(x)
    shared = CustomDenseLayer(64, activation='relu', dropout_rate=0.3, name='custom_dense')(x)

    out_emotion = Dense(NUM_CLASSES, activation='softmax', name='emotion_output')(shared)
    out_healing = Dense(1, activation='linear', name='healing_output')(shared)

    return tf.keras.Model(inputs=inputs, outputs=(out_emotion, out_healing), name='LSTM')


def model_GRU():
    inputs = tf.keras.Input(shape=(MAX_LENGTH,), name='input')
    x      = CustomEmbeddingLayer(vocab_size, EMBED_DIM, name='custom_embedding')(inputs)
    x      = GRU(128, return_sequences=True, name='gru_1')(x)
    x      = Dropout(0.4, name='dropout_1')(x)
    x      = GRU(64, name='gru_2')(x)
    x      = Dropout(0.4, name='dropout_2')(x)
    shared = CustomDenseLayer(64, activation='relu', dropout_rate=0.3, name='custom_dense')(x)

    out_emotion = Dense(NUM_CLASSES, activation='softmax', name='emotion_output')(shared)
    out_healing = Dense(1, activation='linear', name='healing_output')(shared)

    return tf.keras.Model(inputs=inputs, outputs=(out_emotion, out_healing), name='GRU')


def model_CNN():
    inputs = tf.keras.Input(shape=(MAX_LENGTH,), name='input')
    x      = CustomEmbeddingLayer(vocab_size, EMBED_DIM, name='custom_embedding')(inputs)
    x      = CustomConv1DLayer(128, kernel_size=5, dropout_rate=0.3, name='custom_conv_1')(x)
    x      = CustomConv1DLayer(64,  kernel_size=3, dropout_rate=0.3, name='custom_conv_2')(x)
    x      = GlobalMaxPool1D(name='global_max_pool')(x)
    shared = CustomDenseLayer(64, activation='relu', dropout_rate=0.3, name='custom_dense')(x)

    out_emotion = Dense(NUM_CLASSES, activation='softmax', name='emotion_output')(shared)
    out_healing = Dense(1, activation='linear', name='healing_output')(shared)

    return tf.keras.Model(inputs=inputs, outputs=(out_emotion, out_healing), name='CNN')


def model_BiLSTM():
    inputs = tf.keras.Input(shape=(MAX_LENGTH,), name='input')
    x      = CustomEmbeddingLayer(vocab_size, EMBED_DIM, name='custom_embedding')(inputs)
    x      = Bidirectional(LSTM(128, return_sequences=True), name='bilstm_1')(x)
    x      = Dropout(0.4, name='dropout_1')(x)
    x      = Bidirectional(LSTM(64, return_sequences=True), name='bilstm_2')(x)
    x      = Dropout(0.4, name='dropout_2')(x)
    x      = AttentionLayer(units=64, name='attention')(x)
    shared = CustomDenseLayer(64, activation='relu', dropout_rate=0.3, name='custom_dense')(x)

    out_emotion = Dense(NUM_CLASSES, activation='softmax', name='emotion_output')(shared)
    out_healing = Dense(1, activation='linear', name='healing_output')(shared)

    return tf.keras.Model(inputs=inputs, outputs=(out_emotion, out_healing), name='BiLSTM')


def model_BiGRU():
    inputs = tf.keras.Input(shape=(MAX_LENGTH,), name='input')
    x      = CustomEmbeddingLayer(vocab_size, EMBED_DIM, name='custom_embedding')(inputs)
    x      = Bidirectional(GRU(128, return_sequences=True), name='bigru_1')(x)
    x      = Dropout(0.4, name='dropout_1')(x)
    x      = Bidirectional(GRU(64, return_sequences=True), name='bigru_2')(x)
    x      = Dropout(0.4, name='dropout_2')(x)
    x      = AttentionLayer(units=64, name='attention')(x)
    shared = CustomDenseLayer(64, activation='relu', dropout_rate=0.3, name='custom_dense')(x)

    out_emotion = Dense(NUM_CLASSES, activation='softmax', name='emotion_output')(shared)
    out_healing = Dense(1, activation='linear', name='healing_output')(shared)

    return tf.keras.Model(inputs=inputs, outputs=(out_emotion, out_healing), name='BiGRU')

print("✓ Semua model function siap.")

### Train Model

In [ ]:
trainer_LSTM = customTrainingLoop(
    model        = model_LSTM(),
    model_name   = 'LSTM',
    optimizer    = tf.keras.optimizers.Adam(learning_rate=0.001),
    loss_fn      = MultiOutputLoss(emotion_weight=0.7, healing_weight=0.3),
    log_dir      = LOG_DIR,
    class_weight = class_weight,
    callbacks    = [
        CustomEarlyStopping(patience=7, min_delta=0.001),
        OverfittingDetector(threshold=0.15),
        TrainingSummaryLogger(),
        CustomLearningRateScheduler(initial_lr=0.001, decay_factor=0.5, step_size=10),
        CustomModelCheckpoint(
            filepath=os.path.join(OUTPUT_DIR, 'LSTM_checkpoint.keras'),
            monitor='val_loss'
        ),
        CustomTensorBoard(log_dir=LOG_DIR),
    ],
)
trainer_LSTM.fit(train_ds, val_ds, epochs=EPOCH)
trainer_LSTM.plot_history()

In [ ]:
trainer_GRU = customTrainingLoop(
    model        = model_GRU(),
    model_name   = 'GRU',
    optimizer    = tf.keras.optimizers.Adam(learning_rate=0.001),
    loss_fn      = MultiOutputLoss(emotion_weight=0.7, healing_weight=0.3),
    log_dir      = LOG_DIR,
    class_weight = class_weight,
    callbacks    = [
        CustomEarlyStopping(patience=7, min_delta=0.001),
        OverfittingDetector(threshold=0.15),
        TrainingSummaryLogger(),
        CustomLearningRateScheduler(initial_lr=0.001, decay_factor=0.5, step_size=10),
        CustomModelCheckpoint(
            filepath=os.path.join(OUTPUT_DIR, 'GRU_checkpoint.keras'),
            monitor='val_loss'
        ),
        CustomTensorBoard(log_dir=LOG_DIR),
    ],
)
trainer_GRU.fit(train_ds, val_ds, epochs=EPOCH)
trainer_GRU.plot_history()

In [ ]:
trainer_CNN = customTrainingLoop(
    model        = model_CNN(),
    model_name   = 'CNN',
    optimizer    = tf.keras.optimizers.Adam(learning_rate=0.001),
    loss_fn      = MultiOutputLoss(emotion_weight=0.7, healing_weight=0.3),
    log_dir      = LOG_DIR,
    class_weight = class_weight,
    callbacks    = [
        CustomEarlyStopping(patience=7, min_delta=0.001),
        OverfittingDetector(threshold=0.15),
        TrainingSummaryLogger(),
        CustomLearningRateScheduler(initial_lr=0.001, decay_factor=0.5, step_size=10),
        CustomModelCheckpoint(
            filepath=os.path.join(OUTPUT_DIR, 'CNN_checkpoint.keras'),
            monitor='val_loss'
        ),
        CustomTensorBoard(log_dir=LOG_DIR),
    ],
)
trainer_CNN.fit(train_ds, val_ds, epochs=EPOCH)
trainer_CNN.plot_history()

In [ ]:
trainer_BiLSTM = customTrainingLoop(
    model        = model_BiLSTM(),
    model_name   = 'BiLSTM',
    optimizer    = tf.keras.optimizers.Adam(learning_rate=0.001),
    loss_fn      = MultiOutputLoss(emotion_weight=0.7, healing_weight=0.3),
    log_dir      = LOG_DIR,
    class_weight = class_weight,
    callbacks    = [
        CustomEarlyStopping(patience=7, min_delta=0.001),
        StopAtAccuracy(target_acc=0.90),
        OverfittingDetector(threshold=0.15),
        TrainingSummaryLogger(),
        CustomLearningRateScheduler(initial_lr=0.001, decay_factor=0.5, step_size=10),
        CustomModelCheckpoint(
            filepath=os.path.join(OUTPUT_DIR, 'BiLSTM_checkpoint.keras'),
            monitor='val_loss'
        ),
        CustomTensorBoard(log_dir=LOG_DIR),
    ],
)
trainer_BiLSTM.fit(train_ds, val_ds, epochs=EPOCH)
trainer_BiLSTM.plot_history()

In [ ]:
trainer_BiGRU = customTrainingLoop(
    model        = model_BiGRU(),
    model_name   = 'BiGRU',
    optimizer    = tf.keras.optimizers.Adam(learning_rate=0.001),
    loss_fn      = MultiOutputLoss(emotion_weight=0.7, healing_weight=0.3),
    log_dir      = LOG_DIR,
    class_weight = class_weight,
    callbacks    = [
        CustomEarlyStopping(patience=7, min_delta=0.001),
        StopAtAccuracy(target_acc=0.90),
        OverfittingDetector(threshold=0.15),
        TrainingSummaryLogger(),
        CustomLearningRateScheduler(initial_lr=0.001, decay_factor=0.5, step_size=10),
        CustomModelCheckpoint(
            filepath=os.path.join(OUTPUT_DIR, 'BiGRU_checkpoint.keras'),
            monitor='val_loss'
        ),
        CustomTensorBoard(log_dir=LOG_DIR),
    ],
)
trainer_BiGRU.fit(train_ds, val_ds, epochs=EPOCH)
trainer_BiGRU.plot_history()

### Transfer learning

In [ ]:
BERT_MODEL_NAME = 'bert-base-uncased'
BERT_MAX_LENGTH = 128

bert_tokenizer = BertTokenizer.from_pretrained(BERT_MODEL_NAME)

def bert_encode(texts, tokenizer, max_length):
    encoded = tokenizer(
        list(texts),
        max_length     = max_length,
        padding        = 'max_length',
        truncation     = True,
        return_tensors = 'tf'
    )
    return (
        encoded['input_ids'],
        encoded['attention_mask'],
        encoded['token_type_ids']
    )

print("Encoding train...")
train_ids, train_mask, train_type = bert_encode(X_train, bert_tokenizer, BERT_MAX_LENGTH)

print("Encoding val...")
val_ids, val_mask, val_type = bert_encode(X_val, bert_tokenizer, BERT_MAX_LENGTH)

print("Encoding test...")
test_ids, test_mask, test_type = bert_encode(X_test, bert_tokenizer, BERT_MAX_LENGTH)

bert_train_ds = (
    tf.data.Dataset
    .from_tensor_slices((
        {
            'input_ids'     : train_ids,
            'attention_mask': train_mask,
            'token_type_ids': train_type,
        },
        (y_train, yh_train)
    ))
    .shuffle(len(y_train), seed=RANDOM_SEED)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

bert_val_ds = (
    tf.data.Dataset
    .from_tensor_slices((
        {
            'input_ids'     : val_ids,
            'attention_mask': val_mask,
            'token_type_ids': val_type,
        },
        (y_val, yh_val)
    ))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

bert_test_ds = (
    tf.data.Dataset
    .from_tensor_slices((
        {
            'input_ids'     : test_ids,
            'attention_mask': test_mask,
            'token_type_ids': test_type,
        },
        (y_test, yh_test)
    ))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print("✓ bert_train_ds :", len(list(bert_train_ds)), "batch")
print("✓ bert_val_ds   :", len(list(bert_val_ds)),   "batch")
print("✓ bert_test_ds  :", len(list(bert_test_ds)),  "batch")

In [ ]:
def model_BERT():
    input_ids      = tf.keras.Input(shape=(BERT_MAX_LENGTH,), dtype=tf.int32, name='input_ids')
    attention_mask = tf.keras.Input(shape=(BERT_MAX_LENGTH,), dtype=tf.int32, name='attention_mask')
    token_type_ids = tf.keras.Input(shape=(BERT_MAX_LENGTH,), dtype=tf.int32, name='token_type_ids')

    bert_model   = TFBertModel.from_pretrained(BERT_MODEL_NAME)
    bert_outputs = bert_model(
        input_ids,
        attention_mask = attention_mask,
        token_type_ids = token_type_ids,
        training       = False
    )

    cls_output = bert_outputs.last_hidden_state[:, 0, :]  # [CLS] token → (batch, 768)

    shared = Dense(256, activation='relu',  name='shared_dense')(cls_output)
    shared = Dropout(0.3,                   name='shared_dropout')(shared)

    out_emotion = Dense(NUM_CLASSES, activation='softmax', name='emotion_output')(shared)
    out_healing = Dense(1,           activation='linear',  name='healing_output')(shared)

    return tf.keras.Model(
        inputs  = [input_ids, attention_mask, token_type_ids],
        outputs = (out_emotion, out_healing),
        name    = 'BERT'
    )

In [ ]:
class CustomTrainerBERT:
    def __init__(
        self,
        model,
        model_name,
        optimizer,
        loss_fn,
        log_dir      = None,
        class_weight = None,
        callbacks    = None,
    ):
        self.model        = model
        self.model_name   = model_name
        self.optimizer    = optimizer
        self.loss_fn      = loss_fn
        self.log_dir      = log_dir or LOG_DIR
        self.class_weight = class_weight
        self.callbacks    = callbacks or []

        self.history = {
            'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': [],
            'mae': [], 'val_mae': [],
        }

        self.best_val_loss   = np.inf
        self.checkpoint_path = os.path.join(OUTPUT_DIR, f'{model_name}_best.keras')

        self.train_writer = tf.summary.create_file_writer(
            os.path.join(self.log_dir, self.model_name, 'train')
        )
        self.val_writer = tf.summary.create_file_writer(
            os.path.join(self.log_dir, self.model_name, 'val')
        )

        self.train_loss_metric = tf.keras.metrics.Mean(name='train_loss')
        self.train_acc_metric  = tf.keras.metrics.SparseCategoricalAccuracy(name='train_acc')
        self.train_mae_metric  = tf.keras.metrics.MeanAbsoluteError(name='train_mae')
        self.val_loss_metric   = tf.keras.metrics.Mean(name='val_loss')
        self.val_acc_metric    = tf.keras.metrics.SparseCategoricalAccuracy(name='val_acc')
        self.val_mae_metric    = tf.keras.metrics.MeanAbsoluteError(name='val_mae')


    @tf.function
    def _train_step(self, X_batch, y_emotion, y_healing):
        with tf.GradientTape() as tape:
            pred_emotion, pred_healing = self.model(X_batch, training=True)
            loss = self.loss_fn(
                (y_emotion, y_healing),
                (pred_emotion, pred_healing)
            )
            if self.class_weight is not None:
                weights = tf.gather(
                    tf.constant(list(self.class_weight.values()), dtype=tf.float32),
                    tf.cast(y_emotion, tf.int32)
                )
                loss = loss * tf.reduce_mean(weights)

        grads = tape.gradient(loss, self.model.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.model.trainable_variables))

        self.train_loss_metric.update_state(loss)
        self.train_acc_metric.update_state(y_emotion, pred_emotion)
        self.train_mae_metric.update_state(tf.expand_dims(y_healing, -1), pred_healing)


    @tf.function
    def _val_step(self, X_batch, y_emotion, y_healing):
        pred_emotion, pred_healing = self.model(X_batch, training=False)
        loss = self.loss_fn(
            (y_emotion, y_healing),
            (pred_emotion, pred_healing)
        )
        self.val_loss_metric.update_state(loss)
        self.val_acc_metric.update_state(y_emotion, pred_emotion)
        self.val_mae_metric.update_state(tf.expand_dims(y_healing, -1), pred_healing)


    def fit(self, train_ds, val_ds, epochs):
        print(f"\n{'═'*60}")
        print(f"  Model     : {self.model_name}")
        print(f"  Epochs    : {epochs}")
        print(f"  Optimizer : {self.optimizer.__class__.__name__}")
        print(f"  Loss      : {self.loss_fn.__class__.__name__}")
        print(f"  LR        : {float(self.optimizer.learning_rate):.6f}")
        print(f"  Callbacks : {[c.__class__.__name__ for c in self.callbacks]}")
        print(f"{'═'*60}\n")

        for cb in self.callbacks:
            cb.model = self.model
        for cb in self.callbacks:
            if hasattr(cb, 'on_train_begin'):
                cb.on_train_begin(logs={})

        self.total_start = time.time()
        t_loss = 0

        for epoch in range(epochs):
            epoch_start = time.time()

            for X_batch, (y_emotion, y_healing) in train_ds:
                self._train_step(X_batch, y_emotion, y_healing)

            for X_batch, (y_emotion, y_healing) in val_ds:
                self._val_step(X_batch, y_emotion, y_healing)

            t_loss  = self.train_loss_metric.result().numpy()
            t_acc   = self.train_acc_metric.result().numpy()
            t_mae   = self.train_mae_metric.result().numpy()
            v_loss  = self.val_loss_metric.result().numpy()
            v_acc   = self.val_acc_metric.result().numpy()
            v_mae   = self.val_mae_metric.result().numpy()
            elapsed = time.time() - epoch_start

            self.history['loss'].append(t_loss)
            self.history['accuracy'].append(t_acc)
            self.history['mae'].append(t_mae)
            self.history['val_loss'].append(v_loss)
            self.history['val_accuracy'].append(v_acc)
            self.history['val_mae'].append(v_mae)

            with self.train_writer.as_default():
                tf.summary.scalar('loss',     t_loss, step=epoch)
                tf.summary.scalar('accuracy', t_acc,  step=epoch)
                tf.summary.scalar('mae',      t_mae,  step=epoch)

            with self.val_writer.as_default():
                tf.summary.scalar('loss',     v_loss, step=epoch)
                tf.summary.scalar('accuracy', v_acc,  step=epoch)
                tf.summary.scalar('mae',      v_mae,  step=epoch)

            print(f"Epoch {epoch+1:03d}/{epochs} ({elapsed:.1f}s) — "
                  f"loss: {t_loss:.4f}  acc: {t_acc:.4f}  mae: {t_mae:.4f}  "
                  f"val_loss: {v_loss:.4f}  val_acc: {v_acc:.4f}  val_mae: {v_mae:.4f}")

            gap = v_loss - t_loss
            if gap > 0.15:
                print(f" Overfitting — gap: {gap:.4f}")

            if v_loss < self.best_val_loss:
                self.best_val_loss = v_loss
                self.model.save(self.checkpoint_path)
                print(f"✓ Best model saved (val_loss={v_loss:.4f})")

            self.train_loss_metric.reset_states()
            self.train_acc_metric.reset_states()
            self.train_mae_metric.reset_states()
            self.val_loss_metric.reset_states()
            self.val_acc_metric.reset_states()
            self.val_mae_metric.reset_states()

            logs = {
                'loss': t_loss, 'accuracy': t_acc, 'mae': t_mae,
                'val_loss': v_loss, 'val_accuracy': v_acc, 'val_mae': v_mae,
            }
            for cb in self.callbacks:
                if hasattr(cb, 'on_epoch_end'):
                    cb.on_epoch_end(epoch, logs=logs)

            if getattr(self.model, 'stop_training', False):
                print(f"⛔ Training dihentikan oleh callback di epoch {epoch+1}")
                break

        total_elapsed = time.time() - self.total_start

        for cb in self.callbacks:
            if hasattr(cb, 'on_train_end'):
                cb.on_train_end(logs={'loss': t_loss})

        print(f"\n{'═'*60}")
        print(f"  Selesai       : {self.model_name}")
        print(f"  Total waktu   : {total_elapsed:.1f} detik")
        print(f"  Best val_loss : {self.best_val_loss:.4f}")
        print(f"  Checkpoint    : {self.checkpoint_path}")
        print(f"{'═'*60}\n")


    def evaluate(self, test_ds, y_test, yh_test, le):
        print(f"\n{'─'*60}")
        print(f"  Evaluasi : {self.model_name}")
        print(f"{'─'*60}")

        y_pred_emotion_list = []
        y_pred_healing_list = []

        for X_batch, _ in test_ds:
            pred_e, pred_h = self.model(X_batch, training=False)
            y_pred_emotion_list.append(pred_e.numpy())
            y_pred_healing_list.append(pred_h.numpy())

        y_pred_emotion_probs = np.concatenate(y_pred_emotion_list, axis=0)
        y_pred_healing       = np.concatenate(y_pred_healing_list, axis=0).flatten()
        y_pred_emotion       = np.argmax(y_pred_emotion_probs, axis=1)

        acc      = accuracy_score(y_test, y_pred_emotion)
        f1_macro = f1_score(y_test, y_pred_emotion, average='macro')
        f1_w     = f1_score(y_test, y_pred_emotion, average='weighted')
        mae      = np.mean(np.abs(yh_test - y_pred_healing))

        print(f"  Accuracy   : {acc:.4f}  {'✓' if acc >= 0.85 else '✗'} target > 0.85")
        print(f"  F1 Macro   : {f1_macro:.4f}")
        print(f"  F1 Weighted: {f1_w:.4f}")
        print(f"  MAE Healing: {mae:.4f}  {'✓ < 0.2' if mae < 0.2 else '✗ > 0.2'}")
        print(f"\n{classification_report(y_test, y_pred_emotion, target_names=le.classes_)}")

        cm = confusion_matrix(y_test, y_pred_emotion)
        plt.figure(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=le.classes_, yticklabels=le.classes_)
        plt.title(f'Confusion Matrix — {self.model_name}')
        plt.ylabel('True'); plt.xlabel('Predicted')
        plt.tight_layout(); plt.show()

        return {
            'accuracy':    acc,
            'f1_macro':    f1_macro,
            'f1_weighted': f1_w,
            'mae_healing': mae,
        }


    def plot_history(self):
        epochs = range(1, len(self.history['loss']) + 1)

        fig, axes = plt.subplots(1, 3, figsize=(18, 4))
        fig.suptitle(f'Training History — {self.model_name}', fontsize=13)

        axes[0].plot(epochs, self.history['loss'],     label='Train Loss')
        axes[0].plot(epochs, self.history['val_loss'], label='Val Loss')
        axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

        axes[1].plot(epochs, self.history['accuracy'],     label='Train Acc')
        axes[1].plot(epochs, self.history['val_accuracy'], label='Val Acc')
        axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

        axes[2].plot(epochs, self.history['mae'],     label='Train MAE')
        axes[2].plot(epochs, self.history['val_mae'], label='Val MAE')
        axes[2].set_title('MAE Healing Score'); axes[2].legend(); axes[2].grid(True, alpha=0.3)

        plt.tight_layout(); plt.show()


In [ ]:
trainer_BERT = CustomTrainerBERT(
    model        = model_BERT(),
    model_name   = 'BERT',
    optimizer    = tf.keras.optimizers.Adam(learning_rate=2e-5),
    loss_fn      = MultiOutputLoss(emotion_weight=0.7, healing_weight=0.3),
    log_dir      = LOG_DIR,
    class_weight = class_weight,
    callbacks    = [
        CustomEarlyStopping(patience=3, min_delta=0.001),
        OverfittingDetector(threshold=0.15),
        TrainingSummaryLogger(),
        CustomLearningRateScheduler(initial_lr=2e-5, decay_factor=0.5, step_size=2),
        CustomModelCheckpoint(
            filepath=os.path.join(OUTPUT_DIR, 'BERT_checkpoint.keras'),
            monitor='val_loss'
        ),
        CustomTensorBoard(log_dir=LOG_DIR),
    ],
)
trainer_BERT.fit(bert_train_ds, bert_val_ds, epochs=EPOCH)
trainer_BERT.plot_history()

## Evaluasi dan Perbandingan

In [ ]:
all_trainers = {
    'LSTM'  : trainer_LSTM,
    'GRU'   : trainer_GRU,
    'CNN'   : trainer_CNN,
    'BiLSTM': trainer_BiLSTM,
    'BiGRU' : trainer_BiGRU,
}

eval_results = {}
for name, trainer in all_trainers.items():
    eval_results[name] = trainer.evaluate(test_ds, y_test, yh_test, le)

eval_results['BERT'] = trainer_BERT.evaluate(bert_test_ds, y_test, yh_test, le)

eval_df        = pd.DataFrame(eval_results).T.round(4)
eval_df_sorted = eval_df.sort_values('f1_macro', ascending=False)

best_model_name = eval_df_sorted.index[0]

print("\n═══════════════════════════════════════════════════════")
print("   Perbandingan Semua Model")
print("═══════════════════════════════════════════════════════")
print(eval_df_sorted.to_string())
print("═══════════════════════════════════════════════════════")
print(f"\n✓ Model terbaik : {best_model_name}")
print(f"  F1 Macro      : {eval_df_sorted['f1_macro'].iloc[0]:.4f}")
print(f"  Accuracy      : {eval_df_sorted['accuracy'].iloc[0]:.4f}  "
      f"{'✓ > 0.85' if eval_df_sorted['accuracy'].iloc[0] >= 0.85 else '✗ < 0.85'}")
print(f"  MAE Healing   : {eval_df_sorted['mae_healing'].iloc[0]:.4f}  "
      f"{'✓ < 0.2' if eval_df_sorted['mae_healing'].iloc[0] < 0.2 else '✗ > 0.2'}")

# Plot perbandingan 4 metrik
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
fig.suptitle('Perbandingan Semua Model', fontsize=13)

metrics_plot = ['accuracy', 'f1_macro', 'f1_weighted', 'mae_healing']
titles_plot  = ['Accuracy (↑)', 'F1 Macro (↑)', 'F1 Weighted (↑)', 'MAE Healing (↓)']
colors       = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6', '#f39c12', '#1abc9c']
targets      = [0.85, 0.85, 0.85, 0.2]

for ax, metric, title, target in zip(axes, metrics_plot, titles_plot, targets):
    vals = eval_df_sorted[metric]
    bars = ax.bar(eval_df_sorted.index, vals, color=colors[:len(eval_df_sorted)])
    ax.set_title(title)
    ax.set_ylabel('Score')
    ax.set_ylim(0, max(vals.max() * 1.2, 0.1))
    ax.axhline(y=target, color='red', linestyle='--', alpha=0.6, label=f'Target {target}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## Simpan Model

In [ ]:
print("=" * 60)
print("  MENYIMPAN SEMUA MODEL")
print("=" * 60)

all_trainers_full = {**all_trainers, 'BERT': trainer_BERT}

print("\n[1] Simpan semua model → .keras")
for name, trainer in all_trainers_full.items():
    path = os.path.join(OUTPUT_DIR, f'{name}_final.keras')
    trainer.model.save(path)
    print(f"  ✓ {name:8s} → {path}")

print(f"\n[2] Simpan model terbaik ({best_model_name}) → best_model.keras")
best_trainer    = all_trainers_full[best_model_name]
best_keras_path = os.path.join(OUTPUT_DIR, 'best_model.keras')
best_trainer.model.save(best_keras_path)
print(f"  ✓ {best_keras_path}")

print("\n[3] Ekspor semua model → SavedModel format")
for name, trainer in all_trainers_full.items():
    sm_path = os.path.join(SAVED_MODEL_DIR, name)
    trainer.model.export(sm_path)
    print(f"  ✓ {name:8s} → {sm_path}")

print(f"\n[4] Ekspor model terbaik ({best_model_name}) → SavedModel/best")
best_sm_path = os.path.join(SAVED_MODEL_DIR, 'best_model')
best_trainer.model.export(best_sm_path)
print(f"  ✓ {best_sm_path}")

eval_export = {
    name: {k: float(v) for k, v in m.items()}
    for name, m in eval_results.items()
}
eval_export['_meta'] = {
    'best_model'      : best_model_name,
    'best_f1_macro'   : float(eval_df_sorted['f1_macro'].iloc[0]),
    'best_accuracy'   : float(eval_df_sorted['accuracy'].iloc[0]),
    'best_mae_healing': float(eval_df_sorted['mae_healing'].iloc[0]),
}
json_path = os.path.join(OUTPUT_DIR, 'eval_results.json')
with open(json_path, 'w') as f:
    json.dump(eval_export, f, indent=2)

print(f"\n{'═'*60}")
print(f"  ✓ Model terbaik    : {best_model_name}")
print(f"  ✓ Accuracy         : {eval_df_sorted['accuracy'].iloc[0]:.4f}")
print(f"  ✓ F1 Macro         : {eval_df_sorted['f1_macro'].iloc[0]:.4f}")
print(f"  ✓ MAE Healing      : {eval_df_sorted['mae_healing'].iloc[0]:.4f}")
print(f"  ✓ .keras path      : {best_keras_path}")
print(f"  ✓ SavedModel best  : {best_sm_path}")
print(f"  ✓ Eval JSON        : {json_path}")
print(f"{'═'*60}")

## Inference

In [ ]:

def predict_non_bert(text, model, tokenizer, le, max_length=MAX_LENGTH):
    seq    = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_length, padding='post', truncating='post')

    pred_e, pred_h = model.predict(padded, verbose=0)
    probs          = pred_e[0]
    idx            = np.argmax(probs)

    return {
        'text'         : text,
        'prediksi'     : le.classes_[idx],
        'confidence'   : float(probs[idx]),
        'semua'        : {le.classes_[i]: float(p) for i, p in enumerate(probs)},
        'healing_score': float(pred_h[0][0]),
    }


def predict_bert(text, model, bert_tokenizer, le, max_length=BERT_MAX_LENGTH):

    encoded = bert_tokenizer(
        text,
        max_length     = max_length,
        padding        = 'max_length',
        truncation     = True,
        return_tensors = 'tf'
    )
    inputs = {
        'input_ids'     : encoded['input_ids'],
        'attention_mask': encoded['attention_mask'],
        'token_type_ids': encoded['token_type_ids'],
    }
    pred_e, pred_h = model.predict(inputs, verbose=0)
    probs          = pred_e[0]
    idx            = np.argmax(probs)

    return {
        'text'         : text,
        'prediksi'     : le.classes_[idx],
        'confidence'   : float(probs[idx]),
        'semua'        : {le.classes_[i]: float(p) for i, p in enumerate(probs)},
        'healing_score': float(pred_h[0][0]),
    }


def predict_all(text, all_trainers, trainer_bert, tokenizer, bert_tokenizer, le):

    print(f"\n{'═'*65}")
    print(f"  [ALL MODE]")
    print(f"  Input : \"{text}\"")
    print(f"{'═'*65}")

    results = {}

    for name, trainer in all_trainers.items():
        res           = predict_non_bert(text, trainer.model, tokenizer, le)
        results[name] = res
        print(f"\n  [{name}]")
        print(f"    Emosi         : {res['prediksi']} ({res['confidence']:.2%})")
        print(f"    Healing Score : {res['healing_score']:.4f}")
        print(f"    Distribusi    : { {k: f'{v:.2%}' for k, v in res['semua'].items()} }")

    res_bert        = predict_bert(text, trainer_bert.model, bert_tokenizer, le)
    results['BERT'] = res_bert
    print(f"\n  [BERT]")
    print(f"    Emosi         : {res_bert['prediksi']} ({res_bert['confidence']:.2%})")
    print(f"    Healing Score : {res_bert['healing_score']:.4f}")
    print(f"    Distribusi    : { {k: f'{v:.2%}' for k, v in res_bert['semua'].items()} }")

    # Voting majority emosi
    votes       = [r['prediksi'] for r in results.values()]
    majority    = max(set(votes), key=votes.count)
    vote_counts = {label: votes.count(label) for label in le.classes_}

    # Rata-rata healing score semua model
    avg_healing = float(np.mean([r['healing_score'] for r in results.values()]))

    print(f"\n{'─'*65}")
    print(f"  Voting Emosi   : {vote_counts}")
    print(f"  Emosi Final    : {majority} ({votes.count(majority)}/{len(votes)} model sepakat)")
    print(f"  Healing Score  : {avg_healing:.4f} (rata-rata semua model)")
    print(f"{'═'*65}\n")

    return results, majority, avg_healing


def predict_best(text, best_model_name, all_trainers, trainer_bert,
                 tokenizer, bert_tokenizer, le):
    print(f"\n{'═'*65}")
    print(f"  [BEST MODE — {best_model_name}]")
    print(f"  Input : \"{text}\"")
    print(f"{'═'*65}")

    if best_model_name == 'BERT':
        res = predict_bert(text, trainer_bert.model, bert_tokenizer, le)
    else:
        res = predict_non_bert(text, all_trainers[best_model_name].model, tokenizer, le)

    print(f"\n  Emosi         : {res['prediksi']} ({res['confidence']:.2%})")
    print(f"  Healing Score : {res['healing_score']:.4f}")
    print(f"  Distribusi    : { {k: f'{v:.2%}' for k, v in res['semua'].items()} }")
    print(f"{'═'*65}\n")

    return res


# RUN INFERENCE

test_sentences = [
    "I feel so sad and hopeless, nothing seems to go right",
    "Today was absolutely amazing, I am so happy!",
    "I am furious about what just happened, this is unacceptable",
    "I don't know what to feel anymore, everything is numb",
]

# ── ALL MODE ────────────────────────────────────────────────
print("=" * 65)
print("  INFERENCE — ALL MODE (semua 6 model)")
print("=" * 65)

for sentence in test_sentences:
    predict_all(
        text           = sentence,
        all_trainers   = all_trainers,
        trainer_bert   = trainer_BERT,
        tokenizer      = tokenizer,
        bert_tokenizer = bert_tokenizer,
        le             = le
    )

# ── BEST MODE ───────────────────────────────────────────────
print("\n" + "=" * 65)
print(f"  INFERENCE — BEST MODE (model terbaik: {best_model_name})")
print("=" * 65)

for sentence in test_sentences:
    predict_best(
        text            = sentence,
        best_model_name = best_model_name,
        all_trainers    = all_trainers,
        trainer_bert    = trainer_BERT,
        tokenizer       = tokenizer,
        bert_tokenizer  = bert_tokenizer,
        le              = le
    )